In [1]:
from dfft.models.aribtrage import BatteryArbitrage
from dfft.utils.files import get_path
import os
import pandas as pd
import numpy as np
import re
from joblib import Parallel, delayed
import os

#Make sure multi-process doesn't cross threads
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [9]:
def extract_time(d):
    m = re.match(r"(\d{8}_\d{6})", os.path.basename(d))
    return m.group(1) if m else ""

def get_lastest_exp(base_dir):
    exp_dirs = [os.path.join(base_dir, d) for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    exp_dirs = sorted(exp_dirs, key=extract_time)
    return exp_dirs[-1]

def solve_chunk(prices_chunk):
    local_bess = BatteryArbitrage()
    
    chunk_results = []
    for prices in prices_chunk:
        chunk_results.append(local_bess.solve(prices, warm_start=True))
        
    return chunk_results

def solve_dataset(df, num_cores=5, price_key='pred'):
    #Chunk dataset
    chunks = np.array_split(df[price_key].values, num_cores)
    #Multi-process
    results = Parallel(n_jobs=num_cores)(
        delayed(solve_chunk)(chunk) for chunk in chunks
    )
    #Unpack results
    flat_results = [item for sublist in results for item in sublist]
    results_df = pd.DataFrame(flat_results).add_prefix(f'{price_key}_')
    results_df.index = df.index
    return df.join(results_df)

In [3]:
#Define optimization model
bess = BatteryArbitrage()

In [4]:
#Get Latest experiment
base_dir = get_path(r"results\experiments\moment-small\NEMA")
exp_dir = get_lastest_exp(base_dir)

In [5]:
#Read the results
df = pd.read_parquet(exp_dir + "/results.parquet", engine='pyarrow')

In [10]:
df = solve_dataset(df)

In [11]:
print(df)

       split  sample_idx                                               true  \
0      train           0  [17.649999618530273, 15.670000076293945, 9.560...   
1      train           1  [28.559999465942383, 31.06999969482422, 25.840...   
2      train           2  [44.09000015258789, 33.040000915527344, 28.389...   
3      train           3  [24.979999542236328, 23.600000381469727, 21.88...   
4      train           4  [47.88999938964844, 45.02000045776367, 42.8400...   
...      ...         ...                                                ...   
78233   test        8709  [90.31999969482422, 80.61000061035156, 74.7300...   
78234   test        8710  [80.61000061035156, 74.7300033569336, 85.52999...   
78235   test        8711  [74.7300033569336, 85.52999877929688, 74.37000...   
78236   test        8712  [85.52999877929688, 74.37000274658203, 71.4400...   
78237   test        8713  [74.37000274658203, 71.44000244140625, 70.1900...   

                                                   

In [10]:
results = bess.solve(np.array(df.iloc[0]['pred']))
print(f'Est. Profit: ${results["profit"]:.2f}')
print(f'True Profit: ${sum(df.iloc[0]["true"] * (results["discharge_schedule"] - results["charge_schedule"])):.2f}')
results = bess.solve(np.array(df.iloc[0]['true']))
print(f'Optimal Profit: ${results["profit"]:.2f}')

Est. Profit: $25.85
True Profit: $24.50
Optimal Profit: $81.12
